# Neural Network Classifier Implementation


---


### 01. Library Installation


In [ ]:
%pip install -qq matplotlib numpy pandas scikit-learn seaborn tensorflow


### 02. Library Imports


In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf
import seaborn as sns
import pandas as pd
import numpy as np

from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import learning_curve

from pandas.api.types import CategoricalDtype

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve
)


### 03. Data Loading and Preprocessing


This implementation uses the **Breast Cancer Wisconsin (Diagnostic) Dataset** from the UCI Machine Learning Repository for binary classification using neural networks.

### Dataset Characteristics:
- **Total samples**: 699 observations (after removing missing values)
- **Features**: 9 continuous variables describing cell characteristics from fine needle aspirate (FNA) of breast masses
- **Target**: Binary classification (Benign vs Malignant tumors)
  - Class 0: Benign (non-cancerous) - originally labeled as 2
  - Class 1: Malignant (cancerous) - originally labeled as 4

### Medical Context:
The dataset contains digitized images of fine needle aspirate (FNA) of breast masses, describing characteristics of cell nuclei present in the images. Each instance represents measurements taken from a breast mass sample, and the goal is to predict whether the mass is benign or malignant based on these cellular characteristics.

### Feature Descriptions:
| Feature | Description | Range |
|---------|-------------|-------|
| `Clump Thickness` | Thickness of clumps of cells | 1-10 |
| `Uniformity of Cell Size` | Consistency in cell size | 1-10 |
| `Uniformity of Cell Shape` | Consistency in cell shape | 1-10 |
| `Marginal Adhesion` | How cells stick together | 1-10 |
| `Single Epithelial Cell Size` | Size of individual epithelial cells | 1-10 |
| `Bare Nuclei` | Nuclei not surrounded by cytoplasm | 1-10 |
| `Bland Chromatin` | Texture of chromatin in nucleus | 1-10 |
| `Normal Nucleoli` | Size and shape of nucleoli | 1-10 |
| `Mitoses` | Frequency of cell division | 1-10 |

### Class Distribution:
- **Benign (Class 0)**: ~65.5% of samples (458 samples)
- **Malignant (Class 1)**: ~34.5% of samples (241 samples)

The dataset exhibits moderate class imbalance, making it an excellent case study for neural network classification with proper evaluation metrics.

In [ ]:
# Importing and checking the dataframe

dataframe = pd.read_csv('../../datasets/breast_cancer_wisconsin/breast-cancer-wisconsin.data', sep = ',', header = None)
dataframe.head()


In [ ]:
# Renaming columns accordingly to the dataset documentation

columns_name = [
    'Sample code number',
    'Clump Thickness',
    'Uniformity of Cell Size',
    'Uniformity of Cell Shape',
    'Marginal Adhesion',
    'Single Epithelial Cell Size',
    'Bare Nuclei',
    'Bland Chromatin',
    'Normal Nucleoli',
    'Mitoses',
    'Class'
]
dataframe.columns = columns_name
dataframe.head()


In [ ]:
# Checking the possible values for the target variable

# 2: benign (negative)
# 4: malignant (positive)
dataframe['Class'].value_counts()


In [ ]:
# Changing the target variable to numerical values
mapped_values = {
    2: 0,  # benign
    4: 1   # malignant
}

dataframe['Class'] = dataframe['Class'].map(mapped_values)
dataframe['Class'].value_counts()


In [ ]:
# Checking for missing values

dataframe.replace('?', np.nan, inplace = True)
dataframe.isnull().sum()


In [ ]:
# Dropping rows with missing values

dataframe.dropna(inplace = True)
dataframe.isnull().sum()


In [ ]:
# Dropping the first column (Sample code number)

dataframe.drop(columns = ['Sample code number'], inplace = True)
dataframe.head()


In [ ]:
# Normalizing all the feature variables

feature_columns = dataframe.columns[:-1]
dataframe[feature_columns] = dataframe[feature_columns].astype(float)
dataframe[feature_columns] = (dataframe[feature_columns] - dataframe[feature_columns].min()) / (dataframe[feature_columns].max() - dataframe[feature_columns].min())
dataframe.head()


In [ ]:
# Checking all the possible values for all columns

for column in dataframe.columns:
    print(f'Column: {column}')
    print(dataframe[column].value_counts())
    print()


In [ ]:
# Checking the final dataframe information

dataframe.info()


In [ ]:
# Checking the final dataframe data

dataframe.head()


### 04. Data Visualization


In [ ]:
for label in dataframe:
    if label == 'Class' or dataframe[label].dtype == 'object':
        continue

    plt.figure(figsize = (10, 6))

    for class_value in dataframe['Class'].unique():
        subset = dataframe[dataframe['Class'] == class_value]
        sns.kdeplot(subset[label], label = class_value, fill = True, alpha = 0.5)

    plt.title(f'Distribution of {label} by "Class"')
    plt.xlabel(label)
    plt.ylabel('Density')
    plt.legend(title = 'Class')
    plt.grid()

    plt.show()


In [ ]:
# Checking the correlation matrix

plt.figure(figsize = (12, 8))
sns.heatmap(dataframe.corr(), annot = True, fmt = '.2f', cmap = 'coolwarm', square = True)
plt.title('Correlation Matrix')
plt.show()


### 05. Dataset Splitting and Scaling

In [ ]:
# Shuffling the dataset

dataframe = dataframe.sample(frac = 1, random_state = 42).reset_index(drop = True)


In [ ]:
dataframe.info()


In [ ]:
dataframe.head()


In [ ]:
# Defining the train, validation and test datasets sets

train_size = 0.7
validation_size = 0.15
test_size = 0.15


In [ ]:
# Defining the train, validation and test datasets

train_dataset = dataframe[:int(train_size * len(dataframe))]
validation_dataset = dataframe[int(train_size * len(dataframe)):int((train_size + validation_size) * len(dataframe))]
test_dataset = dataframe[int((train_size + validation_size) * len(dataframe)):]

print('\nDatasets sizes:')
print(f'Train dataset size: {len(train_dataset)} samples')
print(f'Validation dataset size: {len(validation_dataset)} samples')
print(f'Test dataset size: {len(test_dataset)} samples')


### 06. Neural Network Classifier Implementation and Evaluation


In [ ]:
# Network Classifier implementation

neural_network_model = tf.keras.Sequential([
    tf.keras.layers.InputLayer(shape = (9,)),

    # First hidden layer
    tf.keras.layers.Dense(64, activation = 'relu'),
    tf.keras.layers.Dropout(0.1),

    # Second hidden layer
    tf.keras.layers.Dense(32, activation = 'relu'),
    tf.keras.layers.Dropout(0.1),

    # Output layer
    # 2 classes, sigmoid for binary classification
    tf.keras.layers.Dense(1, activation = 'sigmoid')
])

neural_network_model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate = 0.001),
    loss = tf.keras.losses.BinaryCrossentropy(),
    metrics = ['accuracy']
)

neural_network_model.summary()


In [ ]:
# Separating features and target variable

X_train = train_dataset.drop(columns = ['Class'])
y_train = train_dataset['Class']

X_test = test_dataset.drop(columns = ['Class'])
y_test = test_dataset['Class']

X_validation = validation_dataset.drop(columns = ['Class'])
y_validation = validation_dataset['Class']

In [ ]:
# Training the model

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor = 'val_loss',
        patience = 10,
        restore_best_weights = True,
        verbose = 1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor = 'val_loss',
        factor = 0.5,
        patience = 5,
        min_lr = 0.001,
        verbose = 1
    )
]

history = neural_network_model.fit(
    X_train,
    y_train,
    validation_data = (X_test, y_test),
    epochs = 100,
    batch_size = 16,
    verbose = 1,
    callbacks = callbacks
)


In [ ]:
# Making predictions

y_probabilities_validation = neural_network_model.predict(X_validation)
y_predictions_validation = (y_probabilities_validation > 0.5).astype(int)

y_probabilities_test = neural_network_model.predict(X_test)
y_predictions_test = (y_probabilities_test > 0.5).astype(int)


In [ ]:
# Classification reports

print('Validation Set Classification Report:')
print(classification_report(y_validation, y_predictions_validation))

print('Test Set Classification Report:')
print(classification_report(y_test, y_predictions_test))


In [ ]:
# Confusion matrix

confusion_matrix_validation = confusion_matrix(y_validation, y_predictions_validation)
confusion_matrix_test = confusion_matrix(y_test, y_predictions_test)

plt.figure(figsize = (8, 6))
sns.heatmap(confusion_matrix_validation, annot = True, fmt = 'd', cmap = 'Blues', cbar = False)
plt.title('Confusion Matrix - Validation Set')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

plt.figure(figsize = (8, 6))
sns.heatmap(confusion_matrix_test, annot = True, fmt = 'd', cmap = 'Greens', cbar = False)
plt.title('Confusion Matrix - Test Set')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()


In [ ]:
# Basic accuracy metrics

accuracy_validation = accuracy_score(y_validation, y_predictions_validation)
accuracy_test = accuracy_score(y_test, y_predictions_test)

print(f'Validation Set Accuracy: {accuracy_validation:.4f}')
print(f'Test Set Accuracy: {accuracy_test:.4f}')


In [ ]:
# Precision metrics

precision_validation = precision_score(y_validation, y_predictions_validation, average = 'weighted')
precision_test = precision_score(y_test, y_predictions_test, average = 'weighted')

print(f'Validation Set Precision (weighted): {precision_validation:.4f}')
print(f'Test Set Precision (weighted): {precision_test:.4f}')

# Class-specific precision
precision_per_class_validation = precision_score(y_validation, y_predictions_validation, average = None)
precision_per_class_test = precision_score(y_test, y_predictions_test, average = None)

print(f'\nValidation Set - Class-specific Precision:')
print(f'\tBenign (0): {precision_per_class_validation[0]:.4f}')
print(f'\tMalignant (1): {precision_per_class_validation[1]:.4f}')

print(f'\nTest Set - Class-specific Precision:')
print(f'\tBenign (0): {precision_per_class_test[0]:.4f}')
print(f'\tMalignant (1): {precision_per_class_test[1]:.4f}')


In [ ]:
# Recall metrics

recall_validation = recall_score(y_validation, y_predictions_validation, average = 'weighted')
recall_test = recall_score(y_test, y_predictions_test, average = 'weighted')

print(f'Validation Set Recall (weighted): {recall_validation:.4f}')
print(f'Test Set Recall (weighted): {recall_test:.4f}')

# Class-specific recall
recall_per_class_validation = recall_score(y_validation, y_predictions_validation, average = None)
recall_per_class_test = recall_score(y_test, y_predictions_test, average = None)

print(f'\nValidation Set - Class-specific Recall:')
print(f'\tBenign (0): {recall_per_class_validation[0]:.4f}')
print(f'\tMalignant (1): {recall_per_class_validation[1]:.4f}')

print(f'\nTest Set - Class-specific Recall:')
print(f'\tBenign (0): {recall_per_class_test[0]:.4f}')
print(f'\tMalignant (1): {recall_per_class_test[1]:.4f}')


In [ ]:
# F1-Score metrics

f1_validation = f1_score(y_validation, y_predictions_validation, average = 'weighted')
f1_test = f1_score(y_test, y_predictions_test, average = 'weighted')

print(f'Validation Set F1-Score (weighted): {f1_validation:.4f}')
print(f'Test Set F1-Score (weighted): {f1_test:.4f}')

# Class-specific F1-Score
f1_per_class_validation = f1_score(y_validation, y_predictions_validation, average = None)
f1_per_class_test = f1_score(y_test, y_predictions_test, average = None)

print(f'\nValidation Set - Class-specific F1-Score:')
print(f'\tBenign (0): {f1_per_class_validation[0]:.4f}')
print(f'\tMalignant (1): {f1_per_class_validation[1]:.4f}')

print(f'\nTest Set - Class-specific F1-Score:')
print(f'\tBenign (0): {f1_per_class_test[0]:.4f}')
print(f'\tMalignant (1): {f1_per_class_test[1]:.4f}')


In [ ]:
# Balanced accuracy

balanced_acc_validation = balanced_accuracy_score(y_validation, y_predictions_validation)
balanced_acc_test = balanced_accuracy_score(y_test, y_predictions_test)

print(f'Validation Set Balanced Accuracy: {balanced_acc_validation:.4f}')
print(f'Test Set Balanced Accuracy: {balanced_acc_test:.4f}')


In [ ]:
# Matthews Correlation Coefficient (MCC)

mcc_validation = matthews_corrcoef(y_validation, y_predictions_validation)
mcc_test = matthews_corrcoef(y_test, y_predictions_test)

print(f'Validation Set Matthews Correlation Coefficient: {mcc_validation:.4f}')
print(f'Test Set Matthews Correlation Coefficient: {mcc_test:.4f}')


In [ ]:
# Cohen's Kappa

kappa_validation = cohen_kappa_score(y_validation, y_predictions_validation)
kappa_test = cohen_kappa_score(y_test, y_predictions_test)

print(f'Validation Set Cohen\'s Kappa: {kappa_validation:.4f}')
print(f'Test Set Cohen\'s Kappa: {kappa_test:.4f}')


In [ ]:
# ROC AUC Score

roc_auc_validation = roc_auc_score(y_validation, y_probabilities_validation)
roc_auc_test = roc_auc_score(y_test, y_probabilities_test)

print(f'Validation Set ROC AUC Score: {roc_auc_validation:.4f}')
print(f'Test Set ROC AUC Score: {roc_auc_test:.4f}')

In [ ]:
# Average Precision Score (PR AUC)

avg_precision_validation = average_precision_score(y_validation, y_probabilities_validation)
avg_precision_test = average_precision_score(y_test, y_probabilities_test)

print(f'Validation Set Average Precision (PR AUC): {avg_precision_validation:.4f}')
print(f'Test Set Average Precision (PR AUC): {avg_precision_test:.4f}')

In [ ]:
# Log Loss (Cross-Entropy Loss)

logloss_validation = log_loss(y_validation, y_probabilities_validation)
logloss_test = log_loss(y_test, y_probabilities_test)

print(f'Validation Set Log Loss: {logloss_validation:.4f}')
print(f'Test Set Log Loss: {logloss_test:.4f}')

In [ ]:
# ROC Curve Visualization

fpr_validation, tpr_validation, _ = roc_curve(y_validation, y_probabilities_validation)
fpr_test, tpr_test, _ = roc_curve(y_test, y_probabilities_test)

plt.figure(figsize = (12, 5))

# Validation ROC Curve
plt.subplot(1, 2, 1)
plt.plot(fpr_validation, tpr_validation, color = 'blue', lw = 2, label = f'ROC Curve (AUC = {roc_auc_validation:.3f})')
plt.plot([0, 1], [0, 1], color = 'red', lw = 1, linestyle = '--', label = 'Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Validation Set')
plt.legend(loc = 'lower right')
plt.grid(alpha = 0.3)

# Test ROC Curve
plt.subplot(1, 2, 2)
plt.plot(fpr_test, tpr_test, color = 'green', lw = 2, label = f'ROC Curve (AUC = {roc_auc_test:.3f})')
plt.plot([0, 1], [0, 1], color = 'red', lw = 1, linestyle = '--', label = 'Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Test Set')
plt.legend(loc = 'lower right')
plt.grid(alpha = 0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Precision-Recall Curve Visualization

precision_validation, recall_validation_curve, _ = precision_recall_curve(y_validation, y_probabilities_validation)
precision_test, recall_test_curve, _ = precision_recall_curve(y_test, y_probabilities_test)

plt.figure(figsize = (12, 5))

# Validation Precision-Recall Curve
plt.subplot(1, 2, 1)
plt.plot(recall_validation_curve, precision_validation, color = 'blue', lw = 2, 
         label = f'PR Curve (AUC = {avg_precision_validation:.3f})')
plt.axhline(y = y_validation.mean(), color = 'red', linestyle = '--', 
            label = f'Random Classifier (AP = {y_validation.mean():.3f})')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve - Validation Set')
plt.legend(loc = 'lower left')
plt.grid(alpha = 0.3)

# Test Precision-Recall Curve
plt.subplot(1, 2, 2)
plt.plot(recall_test_curve, precision_test, color = 'green', lw = 2, 
         label = f'PR Curve (AUC = {avg_precision_test:.3f})')
plt.axhline(y = y_test.mean(), color = 'red', linestyle = '--', 
            label = f'Random Classifier (AP = {y_test.mean():.3f})')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve - Test Set')
plt.legend(loc = 'lower left')
plt.grid(alpha = 0.3)

plt.tight_layout()
plt.show()

### 07. Summary and Interpretation of Additional Metrics

This comprehensive evaluation of our Neural Network classifier on the Breast Cancer Wisconsin dataset provides insights through multiple complementary metrics:

#### **Basic Performance Metrics:**
- **Accuracy**: Measures overall correctness of predictions. Values close to 1.0 indicate excellent performance.
- **Precision**: The proportion of positive predictions that were actually correct. High precision means few false positives.
- **Recall (Sensitivity)**: The proportion of actual positives correctly identified. High recall means few false negatives.
- **F1-Score**: Harmonic mean of precision and recall, providing a single balanced metric.

#### **Advanced Evaluation Metrics:**

**Balanced Accuracy**: Especially important for our moderately imbalanced dataset (65.5% benign, 34.5% malignant). It accounts for class imbalance by averaging recall obtained on each class, making it more reliable than standard accuracy for imbalanced datasets.

**Matthews Correlation Coefficient (MCC)**: Ranges from -1 to +1, with +1 indicating perfect prediction, 0 indicating random prediction, and -1 indicating total disagreement. MCC is particularly valuable for binary classification as it considers all four confusion matrix categories (TP, TN, FP, FN).

**Cohen's Kappa**: Measures inter-rater reliability accounting for chance agreement. Values:
- 0.81-1.00: Almost perfect agreement
- 0.61-0.80: Substantial agreement  
- 0.41-0.60: Moderate agreement
- 0.21-0.40: Fair agreement
- 0.00-0.20: Slight agreement

#### **Probability-Based Metrics:**

**ROC AUC (Area Under ROC Curve)**: Measures the model's ability to distinguish between classes across all classification thresholds. Values close to 1.0 indicate excellent discriminative ability, while 0.5 indicates random performance.

**Average Precision (PR AUC)**: Particularly important for imbalanced datasets like ours. It summarizes the precision-recall curve and is more informative than ROC AUC when dealing with class imbalance, as it focuses on the minority class (malignant cases).

**Log Loss**: Quantifies prediction uncertainty by penalizing confident wrong predictions more heavily. Lower values indicate better calibrated probability estimates.

#### **Medical Context Interpretation:**
In breast cancer diagnosis:
- **False Negatives** (missing malignant cases) are more critical than False Positives
- **High Recall** for malignant class is crucial to avoid missing cancer cases
- **Precision** for malignant class indicates reliability of positive diagnoses
- **ROC curves** show diagnostic accuracy across different decision thresholds
- **PR curves** focus on performance for detecting malignant cases (minority class)

#### **Model Architecture Insights:**
Our neural network uses:
- **Two hidden layers** (64 and 32 neurons) with ReLU activation for non-linear learning
- **Dropout layers** (0.1 rate) to prevent overfitting
- **Sigmoid output** for binary probability estimation
- **Adam optimizer** with learning rate 0.001 for efficient training
- **Early stopping** and **learning rate reduction** to optimize training

The comprehensive metric evaluation demonstrates the model's effectiveness while highlighting areas for potential improvement, particularly in balancing sensitivity and specificity for medical diagnosis applications.